# TMDB review sentiment with the Bing Liu Opinion Lexicon

This notebook scores each review from `DATA/marvel_movie_reviews.csv` using its text and NLTK's Bing Liu Opinion Lexicon. It does **not** train a classifier and does **not** use TMDB ratings.

For each review, the score is: `polarity_score = (positive_word_count - negative_word_count) / total_word_count`, where `total_word_count` is the number of cleaned tokens. Empty or missing reviews receive counts of 0 and a missing (`NaN`) polarity score.

In [1]:
%pip install nltk


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import nltk
import pandas as pd
from nltk.corpus import opinion_lexicon
from nltk.tokenize import RegexpTokenizer


In [3]:
# Find the repository root whether the notebook is launched from the root or SCRIPTS/.
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'DATA').is_dir():
    project_root = project_root.parent

if not (project_root / 'DATA').is_dir():
    raise FileNotFoundError('Could not find the project DATA/ directory.')

input_path = project_root / 'DATA' / 'marvel_movie_reviews.csv'
output_path = project_root / 'OUTPUT' / 'marvel_movie_reviews_lexicon_sentiment.csv'

reviews_df = pd.read_csv(input_path)
if 'review' not in reviews_df.columns:
    raise KeyError("Expected a 'review' column in the input CSV.")

print(f'Loaded {len(reviews_df):,} reviews from {input_path.relative_to(project_root)}')
reviews_df.head()

Loaded 486 reviews from DATA\marvel_movie_reviews.csv


,movie_id,movie,release_date,franchise,review_id,review,review_date,days_since_release
0,557,Spider-Man,2002-05-01,Spider-Man,5c8429f592514127691f8310,Sam Raimi's Spider-Man captures the spirit of ...,2019-03-09,6156
1,557,Spider-Man,2002-05-01,Spider-Man,5dc3308b470ead00158c8738,So many Spiderman movies out there but this wi...,2019-11-06,6398
2,557,Spider-Man,2002-05-01,Spider-Man,5dc38a688d22fc00183d1510,This is one of the few films that you can watc...,2019-11-07,6399
3,557,Spider-Man,2002-05-01,Spider-Man,5dc393ac9d89390015350524,I keep telling people that this is the real Sp...,2019-11-07,6399
4,557,Spider-Man,2002-05-01,Spider-Man,5dc3999b7d2bc100173d1175,Films from the 2000s really are way different ...,2019-11-07,6399


## Load the lexicon

The lists are converted to sets so each cleaned token can be checked directly and transparently.

In [4]:
try:
    nltk.data.find('corpora/opinion_lexicon')
except LookupError:
    nltk.download('opinion_lexicon', quiet=True)

positive_words = set(word.lower() for word in opinion_lexicon.positive())
negative_words = set(word.lower() for word in opinion_lexicon.negative())

print(f'Positive lexicon words: {len(positive_words):,}')
print(f'Negative lexicon words: {len(negative_words):,}')

Positive lexicon words: 2,006
Negative lexicon words: 4,783


## Clean, tokenize, and score reviews

`RegexpTokenizer` retains alphabetic words and contractions while dropping punctuation and numeric-only tokens. All tokens are lowercased before lexicon comparison.

In [5]:
tokenizer = RegexpTokenizer(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def score_review(review):
    """Return positive count, negative count, total word count, and polarity score for one review."""
    if not isinstance(review, str) or not review.strip():
        return 0, 0, 0, float('nan')

    tokens = [token.lower() for token in tokenizer.tokenize(review)]
    total_word_count = len(tokens)

    if total_word_count == 0:
        return 0, 0, 0, float('nan')

    positive_word_count = sum(token in positive_words for token in tokens)
    negative_word_count = sum(token in negative_words for token in tokens)
    polarity_score = (positive_word_count - negative_word_count) / total_word_count

    return positive_word_count, negative_word_count, total_word_count, polarity_score

score_columns = ['positive_word_count', 'negative_word_count', 'total_word_count', 'polarity_score']
reviews_df[score_columns] = pd.DataFrame(
    reviews_df['review'].apply(score_review).tolist(),
    columns=score_columns,
    index=reviews_df.index,
)

reviews_df[score_columns].head()

,positive_word_count,negative_word_count,total_word_count,polarity_score
0,4,0,73,0.054795
1,1,1,32,0.000000
2,1,1,40,0.000000
3,0,0,23,0.000000
4,1,0,26,0.038462


## Inspect the scored reviews

This audit table lets you compare the review text directly with its lexicon counts and normalized polarity score.

In [6]:
display_columns = ['movie', 'review', *score_columns]
reviews_df[display_columns].head(10)

,movie,review,positive_word_count,negative_word_count,total_word_count,polarity_score
0,Spider-Man,Sam Raimi's Spider-Man captures the spirit of ...,4,0,73,0.054795
1,Spider-Man,So many Spiderman movies out there but this wi...,1,1,32,0.000000
2,Spider-Man,This is one of the few films that you can watc...,1,1,40,0.000000
3,Spider-Man,I keep telling people that this is the real Sp...,0,0,23,0.000000
4,Spider-Man,Films from the 2000s really are way different ...,1,0,26,0.038462
5,Spider-Man,You can take your Tom Holland and Andrew Garfi...,1,0,29,0.034483
6,Spider-Man,It's 2019 and I still get chills every time I ...,0,0,12,0.000000
7,Spider-Man,Truly one of the most successful superheroes e...,1,0,9,0.111111
8,Spider-Man,"Was, still is, and will forever be one of my f...",1,0,14,0.071429
9,Spider-Man,Lot of respect for Tobey Maguire for doing wel...,3,0,33,0.090909


## Polarity-score summary statistics

In [7]:
print('Overall polarity-score statistics:')
display(reviews_df['polarity_score'].describe())

empty_score_count = reviews_df['polarity_score'].isna().sum()
print(f'Reviews with no scorable tokens: {empty_score_count:,}')

movie_summary = (
    reviews_df.groupby('movie', dropna=False)
    .agg(
        review_count=('review', 'size'),
        scored_review_count=('polarity_score', 'count'),
        mean_polarity_score=('polarity_score', 'mean'),
        median_polarity_score=('polarity_score', 'median'),
        min_polarity_score=('polarity_score', 'min'),
        max_polarity_score=('polarity_score', 'max'),
    )
    .sort_values('mean_polarity_score', ascending=False)
)

movie_summary

Overall polarity-score statistics:


count    486.000000
mean       0.056205
std        0.118865
min       -0.500000
25%        0.000000
50%        0.037037
75%        0.076923
max        1.000000
Name: polarity_score, dtype: float64

Reviews with no scorable tokens: 0


,review_count,scored_review_count,mean_polarity_score,median_polarity_score,min_polarity_score,max_polarity_score
movie,,,,,,
Avengers: Infinity War,33,33,0.154186,0.067114,-0.036364,1.000000
Spider-Man: Homecoming,11,11,0.126081,0.034483,-0.019022,1.000000
Spider-Man: Far From Home,16,16,0.113409,0.066803,-0.031250,0.500000
Avengers: Age of Ultron,40,40,0.070839,0.050894,-0.018868,0.304348
Captain America: The First Avenger,9,9,0.066095,0.052369,0.021622,0.136364
The Avengers,41,41,0.062570,0.065728,-0.022222,0.150000
Spider-Man: No Way Home,16,16,0.059177,0.048852,-0.026385,0.333333
Captain America: The Winter Soldier,13,13,0.058575,0.041667,0.008646,0.200000
Spider-Man: Into the Spider-Verse,53,53,0.057103,0.062500,-0.500000,0.333333


## Export the enriched review data

The raw CSV in `DATA/` is not changed. The new file contains every original column plus the three sentiment columns.

In [8]:
output_path.parent.mkdir(parents=True, exist_ok=True)
reviews_df.to_csv(output_path, index=False)